In [ ]:
#Importing required libraries
#Importing pandas to read and analyze data
import pandas as pd
#Importing numpy library to perform math operations like mean, sum, matrix multiplication
import numpy as np
#Importing plotting library to create charts and graphs
import matplotlib.pyplot as plt
#Importing library to create statistical graphs
import seaborn as sns
!pip install -U fg-data-profiling





In [ ]:
#Data importing and verifying
#This is the file path to excel file (data set)
file_path="../dataset/Sales Data_PDA_4052.xlsx"
#This is DataFrame (df) to read excel file (data set) -
df = pd.read_excel(file_path)

In [ ]:
#Preview data - usually first 5 rows with column names
df.head()

In [ ]:
#Preview data - usually last 5 rows with column names
df.tail()

In [ ]:
#Data Understanding and Preparation
#Detecting missing values, data types, data structure, and data size
df.info()

In [ ]:
#Finding outliers and understanding data distriibution with mean, min, max, standard deviation (std), max, and quartiles (25%, 50%, 75%)
df.describe()

In [ ]:
#Finding the dimensions (rows, columns) of a Pandas DataFrame
df.shape

In [ ]:
#checking data types
df.dtypes

In [ ]:
#finding missing values in each column of dataset
df.isnull().sum()

In [ ]:
#Finding all columns with missing values
[items for items in df.columns if df[items].isnull().sum()>0]

In [ ]:
#Data Cleaning
#Dimensions BEFORE cleaning
print("\nDimensions Before Cleaning Data:", df.shape)

In [ ]:
#Making first row as header
df = pd.read_excel(file_path, header=1)
df.head()

In [ ]:
#Verifying header and all information
df.info()

In [ ]:
#verifying the updated data types
df.dtypes

In [ ]:
#Finding outliers and understanding data distribution with mean, min, max, standard deviation (std), max, and quartiles (25%, 50%, 75%)
df.describe()

In [ ]:
#Remove rows containing missing (NaN) value if any
df.dropna()

In [ ]:
#Checking number of duplicate values in the dataset
df.duplicated().sum()

In [ ]:
#Remove duplicate rows/values from the dataset to ensure data consistency
df = df.drop_duplicates()

In [ ]:
#Trailing space detection
mask = df.apply(lambda col: col.astype(str).str.contains(r"\s+$", na=False))
count = mask.sum().sum()
print(count)

In [ ]:
#remove trailing space
df = df.apply(lambda col: col.str.rstrip() if col.dtype == "object" else col)

In [ ]:
#Outlier Detection (Before)
plt.figure()
df["value_£"].hist(bins=30)
plt.title("Sales Distribution Before Outlier Removal")
#Saving the histogram as an image file for reporting and documentation purposes
plt.savefig("value_hist_before_outliers.png", dpi=300, bbox_inches="tight")
#Displaying the histogram plot
plt.show()

In [ ]:
# Removing outliers using IQR method
Q1 = df["value_£"].quantile(0.25)
Q3 = df["value_£"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
df = df[(df["value_£"] >= lower_bound) & (df["value_£"] <= upper_bound)]
print(df)

In [ ]:
# After outlier removal
plt.figure()
df["value_£"].hist(bins=30)
plt.title("Sales Distribution After Outlier Removal")
plt.savefig("sales_distribution_after_outliers.png", dpi=300, bbox_inches="tight")
plt.show()

#Dimensions AFTER cleaning
print("\nDimensions After Cleaning Data:", df.shape)

In [ ]:
#Data Transformation and Feature Engineering
#Standardising text
df["sales_person"] = df["sales_person"].str.strip().str.title()
df["priority"] = df["priority"].str.strip().str.title()
df["ship_mode"] = df["ship_mode"].str.strip().str.title()
#Renaming column
df = df.rename(columns={"value_£": "sales_values"})
print(df)

In [ ]:
#Mapping categorical priority to numerical scale
priority_map = {
    "Not Specified": 0,
    "Low": 1,
    "Medium": 2,
    "High": 3,
    "Critical": 4
}
df["priority_score"] = df["priority"].map(priority_map)
print(df)

In [ ]:
#Creating new feature combining priority and sales
df["priority_values"] = df["priority_score"] * df["sales_values"]
print(df)

In [ ]:
#Adding the currency column
df["currency"] = "GBP"
print(df)

In [ ]:
#Converting date
df["date"] = pd.to_datetime(df["date"])

#Creating time features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_name"] = df["date"].dt.day_name()
df["month_name"] = df["date"].dt.month_name()
df["week"] = df["date"].dt.isocalendar().week
#Defining month order for chronological analysis
month_order = ["January","February","March","April","May","June",
               "July","August","September","October","November","December"]
#Creating a feature to identify weekend transactions
df["is_weekend"] = df["day_name"].isin(["Saturday", "Sunday"])
#Splitting sales values into groups using defined bins
df["sales_category"] = pd.cut(
    df["sales_values"],
    bins=[0, 500, 2000, 10000, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)
print(df)

In [ ]:
#Calculating average sales per customer using groupby transformation
df["customer_avg_sales"] = df.groupby("customer_id")["sales_values"].transform("mean")
print (df)

In [ ]:
#Creating a binary feature for high value orders based on mean sales
df["high_value_order"] = df["sales_values"] > df["sales_values"].mean()
#Calculating total spend per customer using group aggregation
df["customer_total_spend"] = df.groupby("customer_id")["sales_values"].transform("sum")
#Calculating total sales per salesperson using groupby transformation
df["salesperson_total_sales"] = df.groupby("sales_person")["sales_values"].transform("sum")
print(df)

In [ ]:
#Applying log transformation to reduce skewness in sales distribution
df["log_value"] = np.log1p(df["sales_values"])
#Creating binary feature based on median sales threshold
df["high_value"] = (df["sales_values"] > df["sales_values"].median()).astype(int)
#Calculating time difference between consecutive orders per customer
df["days_since_last_order"] = df.groupby("customer_id")["date"].diff().dt.days
print(df)

In [ ]:
#Sorting data by customer_id to support time-based calculations
df = df.sort_values(by="customer_id")
#Sorting data for time-based analysis per customer
df = df.sort_values(["customer_id", "date"])
#Calculating average sales grouped by priority level
df.groupby("priority")["sales_values"].mean()



In [ ]:
#Customer insights
df.groupby("customer_id").size()
df.groupby("customer_id")["sales_values"].sum()
#Identifying top 10 most frequent customers (Basic EDA)
df["customer_id"].value_counts().head(10)

customer_summary = df.groupby("customer_id").agg(
    total_orders=("order_id", "count"),
    total_spent=("sales_values", "sum")
)
print(customer_summary)

In [ ]:
#Verifying outliers and verifying data distribution in updated dataset
df.describe()

In [ ]:
#Examining the frequency distribution
# Frequency Distribution Plots
fig, axes = plt.subplots(1, 3, figsize=(18,5))
#Shipping modes to understand delivery patterns
df["ship_mode"].value_counts().plot(kind="bar", ax=axes[1])
axes[1].set_title("Shipping Mode Distribution")
axes[1].set_xlabel("Ship Mode")
axes[1].set_ylabel("Count")
#Sales representatives to identify workload distribution
df["sales_person"].value_counts().plot(kind="bar", ax=axes[2])
axes[2].set_title("Salesperson Distribution")
axes[2].set_xlabel("Salesperson")
axes[2].set_ylabel("Count")
#Priority levels to understand category balance
df["priority"].value_counts().plot(kind="bar", ax=axes[0])
axes[0].set_title("Priority Distribution")
axes[0].set_xlabel("Priority")
axes[0].set_ylabel("Count")

plt.tight_layout()
plt.savefig("Frequency_distribution_plots.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Calculating total sales in dataset
df["sales_values"].sum()

In [ ]:
#Aggregations
#Total sales by salesperson
display(df.groupby("sales_person")["sales_values"].sum().sort_values(ascending=False))
#Average sales by shipping mode
display(df.groupby("ship_mode")["sales_values"].mean())
#Average sales by priority
display(df.groupby("priority")["sales_values"].mean())
#Monthly total sales
display(df.groupby("month_name")["sales_values"].sum())
#Total sales interms of shipping mode
display(df.groupby("ship_mode")["sales_values"].sum())

In [ ]:
#Identifying top 10 most frequent customers
top_customers = df["customer_id"].value_counts().head(10)
#Plotting graph
plt.figure(figsize=(8,5))
top_customers.plot(kind="bar")
plt.title("Top 10 Most Frequent Customers")
plt.xlabel("Customer ID")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("top_10_customer_by_spending.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Top 10 customers by total sales
top_customers_sales = (df.groupby("customer_id")["sales_values"].sum().sort_values(ascending=False).head(10))
#Plot graph
plt.figure(figsize=(8,5))
top_customers_sales.plot(kind="bar")
plt.title("Top 10 Customers by Total Sales")
plt.xlabel("Customer ID")
plt.ylabel("Total Sales Value")
plt.xticks(rotation=45)
plt.tight_layout()
# Save graph
plt.savefig("top10_customers_total_sales.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Total Sales by Salesperson
salesperson_total = df.groupby("sales_person")["sales_values"].sum().sort_values(ascending=False)
print(salesperson_total)
salesperson_total.plot(kind="bar")
plt.title("Total Sales by Salesperson")
plt.xlabel("Salesperson")
plt.ylabel("Total Sales Value")
plt.xticks(rotation=0)
plt.savefig("total_sales_by_salesperson.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Boxplot (Sales vs Priority)
plt.figure(figsize=(10,6))
sns.boxplot(x="priority", y="sales_values", data=df)
plt.title("Sales Value by Priority", pad=15)
plt.xlabel("Priority")
plt.ylabel("Sales Value")
#Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("sales_priority_boxplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Relationships
#Sales Person vs Priority
ct = pd.crosstab(df["sales_person"], df["priority"])
ax = ct.plot(kind="bar", stacked=True, figsize=(10,6))
plt.title("Sales Person vs Priority", pad=15)
plt.xlabel("Sales Person")
plt.ylabel("Number of Orders")
#Moving legend outside
ax.legend(title="Priority", bbox_to_anchor=(1.05, 1), loc='upper left')
#Adding bar labels
for container in ax.containers: ax.bar_label(container, fmt='%.0f', padding=3)
#Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("salesperson_priority_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Sales value by Sales Person and Priority
ax = df.groupby(["sales_person", "priority"])["sales_values"] \
       .sum().unstack().plot(kind="bar", stacked=True, figsize=(10,6))
plt.title("Sales Value by Sales Person and Priority", pad=15)
plt.xlabel("Sales Person")
plt.ylabel("Sales Value")
#Moving legend outside
ax.legend(title="Priority", bbox_to_anchor=(1.05, 1), loc='upper left')
#Adding bar labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
#Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("salesvalues_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Sales Person + Ship Mode vs Priority
grouped = df.groupby(["sales_person", "ship_mode", "priority"]).size().unstack()
ax = grouped.plot(kind="bar", stacked=True, figsize=(10,6))
plt.title("Sales Person + Ship Mode vs Priority")
plt.xlabel("Sales Person and Shipping Mode")
plt.ylabel("Number of Orders")
#Moving legend outside
ax.legend(title="Priority", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("salesvalues_ship_priro_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Priority vs time heatmap
for person in df["sales_person"].unique():
    temp = df[df["sales_person"] == person]
    table = pd.crosstab(temp["month_name"], temp["priority"])
    plt.figure(figsize=(8,4))
    sns.heatmap(table, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{person} - Month vs Priority")
    plt.savefig(f"month_priority_{person}.png", dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
#Priority numeric relation
plt.figure()
plt.scatter(df["priority_score"], df["sales_values"])
plt.title("Priority Level and Sales Value")
plt.xlabel("Priority Level")
plt.ylabel("Sales Value")
plt.savefig("num_sales_priority.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Correlation between priority and sales value
correlation = df["priority_score"].corr(df["sales_values"])
print("Correlation between Priority and Sales Value:", correlation)
plt.figure()
sns.heatmap(df[["priority_score", "sales_values"]].corr(), annot=True)
plt.title("Correlation Between Priority and Sales Value")
plt.savefig("priority_sales_correlation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Monthly shipping analysis
table = df.groupby(["month_name", "ship_mode"]).size().unstack()
ax = table.plot(kind="bar", stacked=True, figsize=(10,6))
plt.title("Ship Mode Distribution by Month", pad=15)
ax.legend(title="Ship Mode", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.ylabel("No. of Orders")
plt.xlabel("Shipping Month")
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
plt.tight_layout()
plt.savefig("shipping.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Monthly revenue analysis
monthly_revenue = df.groupby("month_name")["sales_values"].sum()
monthly_revenue = monthly_revenue.reindex(month_order)
print("Highest Month:", monthly_revenue.idxmax())
print("Revenue:", monthly_revenue.max())
ax = monthly_revenue.plot(kind="bar", figsize=(10,6))
plt.title("Monthly Revenue", pad=15)
plt.xlabel("Month Name")
plt.ylabel("Sales Value")
plt.xticks(rotation=45)
#Adding bar labels
for container in ax.containers: ax.bar_label(container, fmt='%.0f', padding=3)
#Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("monthly_revenue.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Weekly revenue analysis
weekly_rev = df.groupby(["week", "sales_person"])["sales_values"].sum().reset_index()
plt.figure(figsize=(10,6))
for person in weekly_rev["sales_person"].unique():
    data = weekly_rev[weekly_rev["sales_person"] == person]
    plt.plot(data["week"], data["sales_values"], marker="o", label=person)
plt.title("Weekly Revenue per Sales Person", pad=15)
plt.ylabel("Sales Value")
plt.xlabel("Number of Week")
# Moving legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
# Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("salesperson_weekly.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Unique customer Per Sales person
summary = df.groupby("sales_person").agg(
    unique_customers=("customer_id", "nunique"),
    total_sales=("sales_values", "sum")
)
print(summary)
ax = summary["unique_customers"].sort_values().plot(kind="bar", figsize=(10,6))
plt.title("Unique Customers per Sales Person", pad=15)
plt.xlabel("Sales Person")
plt.ylabel("Number of Customers")
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
plt.tight_layout()
plt.savefig("salesperson_analysis.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Monthly sales per person
monthly_sales = df.groupby(["sales_person", "month_name"])["sales_values"].sum().reset_index()
monthly_sales["month_name"] = pd.Categorical(
    monthly_sales["month_name"],
    categories=month_order,
    ordered=True
)
monthly_sales = monthly_sales.sort_values("month_name")
plt.figure(figsize=(10,6))
for person in monthly_sales["sales_person"].unique():
    data = monthly_sales[monthly_sales["sales_person"] == person]
    plt.plot(data["month_name"], data["sales_values"], marker="o", label=person)
plt.title("Monthly Revenue per Sales Person", pad=15)
plt.xlabel("Month Name")
plt.ylabel("Sales Value")
# Moving legend outside
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.savefig("salesperson_monthly.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
#Best month per sales person

monthly_best = df.groupby(["sales_person", "month_name"])["sales_values"].sum().reset_index()
best_months = monthly_best.loc[
    monthly_best.groupby("sales_person")["sales_values"].idxmax()
]
print(best_months)
fig, ax = plt.subplots(figsize=(10,6))
ax.bar(best_months["sales_person"], best_months["sales_values"])
plt.title("Best Month per Salesperson", pad=15)
plt.xlabel("Sale Person")
plt.ylabel("Sales Value")
#Adding bar labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3)
#Adjusting layout to avoid cutting
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.savefig("best_month.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
#Importing ProfileReport to generate an automated exploratory data analysis (EDA) report
from data_profiling import ProfileReport
#Creating a profiling report for the cleaned dataset
profile = ProfileReport(df)
#Displaying the interactive HTML report within the notebook
profile.to_notebook_iframe()